[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/28_moe_solution.ipynb)

# 🔴 Solution: Mixture of Experts (MoE)（参考版）

## 📋 题目描述

**难度：** Hard

实现混合专家（MoE）层（nn.Module）。

MoE 通过学习的路由器将每个 token 路由到 top-k 个专家，用 softmax 归一化的权重组合它们的输出，实现条件计算。

**签名:** `MixtureOfExperts(d_model, d_ff, num_experts, top_k=2)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (B, S, d_model)

**返回:** 输出张量 (B, S, d_model)

**约束:**
- 路由器：`nn.Linear(d_model, num_experts)` -> topk -> softmax
- 每个专家：Linear -> ReLU -> Linear
- 专家存储在 `self.experts`（nn.ModuleList）中

**提示：** `router`：`Linear(d, num_experts)` → `topk` → `softmax` 权重。每个专家：`Linear → ReLU → Linear`。对每个 token 的 top-k 专家输出加权求和。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k

        # ---- Step 1: 路由器（门控网络）----
        # 路由器是一个线性层，输出每个专家的"得分"
        # 输入: (batch, d_model) → 输出: (batch, num_experts)
        # 得分经过 topk + softmax 决定每个 token 去哪几个专家
        self.router = nn.Linear(d_model, num_experts)

        # ---- Step 2: 专家网络 ----
        # 每个专家是一个独立的 FFN：Linear → ReLU → Linear
        # d_model → d_ff → d_model
        # nn.ModuleList 存储所有专家，像列表一样索引
        # 总参数量 = num_experts × (d_model*d_ff + d_ff + d_ff*d_model + d_model)
        # 但每次只激活 top_k 个专家 → 条件计算
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        orig_shape = x.shape
        # ---- Step 3: 展平为 2D ----
        # MoE 对每个 token 独立路由，所以先把 (B,S,D) 展平为 (N,D)
        if x.dim() == 3:
            B, S, D = x.shape
            x_flat = x.reshape(-1, D)
        else:
            x_flat = x
        N = x_flat.shape[0]

        # ---- Step 4: 路由 ----
        # logits = router(x_flat): (N, num_experts)
        # topk: 取得分最高的 top_k 个专家
        #   top_vals: (N, top_k) — 选中专家的得分
        #   top_idx:  (N, top_k) — 选中专家的索引
        logits = self.router(x_flat)
        top_vals, top_idx = logits.topk(self.top_k, dim=-1)

        # ---- Step 5: 归一化权重 ----
        # 只对选中的 top_k 个专家得分做 softmax
        # 这样每个 token 的 top_k 个专家权重之和 = 1
        weights = torch.softmax(top_vals, dim=-1)  # (N, top_k)

        # ---- Step 6: 加权组合专家输出 ----
        # 遍历 top_k 和 num_experts，对每个 (专家, 位置) 组合计算
        # mask 选出"第 k 个路由选择了专家 e"的 token
        # 用 masked 索引高效计算
        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                # 找到第 k 个路由选择了专家 e 的所有 token
                mask = (top_idx[:, k] == e)
                if mask.any():
                    # weights[mask, k:k+1]: 保持二维，方便广播
                    # self.experts[e](x_flat[mask]): 专家 e 对这些 token 的输出
                    output[mask] += weights[mask, k:k+1] * self.experts[e](x_flat[mask])

        return output.reshape(orig_shape)


In [ ]:
moe = MixtureOfExperts(d_model=64, d_ff=128, num_experts=4, top_k=2)
x = torch.randn(2, 10, 64)
out = moe(x)
print(f'Output: {out.shape}')
print(f'Total params: {sum(p.numel() for p in moe.parameters())}')


In [ ]:
from torch_judge import check
check('moe')


## 📝 核心思路总结

1. **路由器**：Linear(d, num_experts) + topk + softmax 决定每个 token 去哪些专家
2. **专家网络**：每个专家是独立的 FFN（Linear→ReLU→Linear）
3. **条件计算**：每个 token 只激活 top_k 个专家，计算量与专家数无关
4. **加权组合**：softmax 归一化后的权重 × 专家输出，求和得到最终结果
